# Introduction

This notebook is part of the Machine Learning lifecycle for the **Melbourne Housing Market** dataset.  
The goal of this stage is to prepare the dataset for modeling by performing:

- Data preprocessing (dropping irrelevant features, handling outliers, missing values, etc.)
- Feature engineering (encoding, scaling, transformations)
- Building reproducible **pipelines** for training and inference

We assume that data cleaning and EDA have already been completed.


In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# ML tools
from sklearn.model_selection import cross_val_score, KFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import (
    VarianceThreshold,
    RFECV,
    SequentialFeatureSelector
)
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.feature_selection import f_regression, mutual_info_regression,SelectKBest

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## Load the Dataset

We load the cleaned dataset obtained after EDA.  
Ensure the file path points to the latest cleaned version of the dataset.


In [ ]:
df_original = pd.read_csv('data/melbourne_housing_cleaned_v1.csv')
df_original.head()

## Drop Non-Significant Columns

From the EDA stage, we identified certain columns that have little or no predictive power or contain excessive unique values. We will remove those columns to simplify the model and reduce noise.


In [ ]:
cols_to_drop = ["Address","ParkingArea","Rooms","Postcode", "Date"]

df_original.drop(columns=cols_to_drop, inplace=True)
print(f"Dropped columns: {cols_to_drop}")

## Handle Outliers

After reviewing the EDA results and visualizing distributions using box plots:

- The columns **`Landsize`**, **`BuildingArea`**, and **`Price`** appeared to be**highly skewed** with extreme right-side values.  
  We will apply a **log transformation** to these columns to reduce skewness and normalize their distributions.

- The columns **`Distance`**, **`Car`**, and **`Bathroom`**, though slightly skewed, do **not exhibit extreme outliers**.  
  Their value ranges (max values around 49, 7, and 7 respectively) are reasonable, so we will **retain them as-is**.

- The **`YearBuilt`** column, though skewed, naturally reflects the distribution of house construction years  
  and will also be left unchanged.

> *Note:* Because log(0) is undefined, we will add +1 to each column before applying the transformation to handle any zero values safely.



In [ ]:
has_nan = df_original.isna().any().any()
print("Contains NaN values:", has_nan)

In [ ]:
# Columns identified as highly skewed
skewed_cols = ['Landsize', 'BuildingArea']

# Apply log(1 + x) transformation to reduce skewness
for col in skewed_cols:
    if col in df_original.columns:
        df_original[f'Log_{col}'] = np.log1p(df_original[col])  # log1p handles 0 values gracefully

In [ ]:
for col in skewed_cols:
    plt.figure(figsize=(12, 5))

    # Original
    plt.subplot(1, 2, 1)
    sns.histplot(df_original[col], bins=50, kde=True, color='skyblue')
    plt.title(f'Original {col}')
    plt.xlabel(col)

    # Log-transformed
    plt.subplot(1, 2, 2)
    sns.histplot(df_original[f'Log_{col}'], bins=50, kde=True, color='lightgreen')
    plt.title(f'Log-Transformed {col}')
    plt.xlabel(f'Log_{col}')

    plt.tight_layout()
    plt.show()

    # Print skewness before and after
    original_skew = df_original[col].skew()
    log_skew = df_original[f'Log_{col}'].skew()
    print(f"{col} -> Skewness before: {original_skew:.2f}, after log-transform: {log_skew:.2f}\n")

## FEATURE ENGINEERING
Feature engineering aims to enhance the dataset by transforming existing variables into more informative or model-friendly forms.  
In this stage, we perform **feature transformation**, focusing primarily on encoding categorical variables so they can be effectively used in machine learning models.

Different categorical features have different characteristics — some have only a few possible values (low cardinality), while others have hundreds (high cardinality).  
Hence, a **mixed encoding strategy** is used to handle each feature appropriately.

### 1. Feature Transformation
Different encoding strategies are used based on the **cardinality** and **semantic meaning** of each categorical variable:

| Feature | Unique | Encoding Method | Reason |
|----------|----------|----------------|--------|
| Suburb | 295 | Frequency / Target | High cardinality, location-based |
| SellerG | 195 | Frequency / Target | High cardinality, weak effect |
| Type | 3 | One-Hot | Low cardinality, categorical |
| Method | 5 | One-Hot | Low cardinality, sale type |
| CouncilArea | 32 | Frequency / Target | Medium cardinality, location |
| Regionname | 8 | One-Hot | Low cardinality, regional grouping |

This mixed encoding approach balances interpretability and efficiency, preventing excessive feature expansion while preserving meaningful relationships.

In [ ]:
encoded_df = df_original.copy()

# ====== 1. Frequency Encoding ======
# Suitable for features with many unique values like Suburb, SellerG, CouncilArea
for col in ["Suburb", "SellerG","CouncilArea"]:
    freq_map = encoded_df[col].value_counts().to_dict()
    encoded_df[f"{col}_enc"] = encoded_df[col].map(freq_map)
    print(f"Applied frequency encoding to '{col}'")

# ====== 2. One-Hot Encoding ======
# For low-cardinality categorical features
onehot_features = ["Type", "Method", "Regionname"]
onehot_encoder = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')

# Fit and transform
onehot_encoded = onehot_encoder.fit_transform(encoded_df[onehot_features])

# Get feature names
feature_names = onehot_encoder.get_feature_names_out(onehot_features)

# Create DataFrame with encoded features
onehot_df = pd.DataFrame(onehot_encoded, columns=feature_names, index=encoded_df.index)

# Drop original columns and concatenate encoded ones
encoded_df = encoded_df.drop(columns=onehot_features)
encoded_df = pd.concat([encoded_df, onehot_df], axis=1)

print(f"Applied one-hot encoding to {onehot_features}")

# Drop encoded columns
encoded_df = encoded_df.drop(columns=['Suburb', 'SellerG', 'CouncilArea'])

In [ ]:
encoded_df.head()

 1. **Frequency Encoding**
   - Replaces each category with its occurrence count.
   - Retains useful information about category prevalence without inflating feature space.
   - Example: If *Suburb A* appears 300 times and *Suburb B* appears 10 times, their encoded values will be 300 and 10.

 2. **Target Encoding**
   - Replaces each category with the mean value of the target (`Price`) within that category.
   - Captures the direct relationship between categorical values and the target variable.
   - Applied carefully to avoid data leakage (ideally within cross-validation in the full pipeline).

 3. **One-Hot Encoding**
   - Expands low-cardinality categorical features into separate binary columns.
   - Keeps interpretability while maintaining model compatibility.
   - Used for `Type`, `Method`, and `Regionname`, which each have only a few possible values.


In [ ]:
has_nan = encoded_df.isna().any().any()
print("Contains NaN values:", has_nan)

In [ ]:
encoded_df.shape

### 2. Feature Creation

Feature creation involves generating new variables derived from existing ones to expose hidden patterns or improve model performance.  
For housing data, this step often includes combining, scaling, or deriving features that better reflect property characteristics and market dynamics.

In this dataset, we create new features that describe:
- **Age of the property**
- **Land and building proportions**
- **Distance-based attributes**

These engineered features help the model understand structural efficiency, property condition, and location context.


In [ ]:
features_df = df_original.copy()

# --- Property Age ---
# Convert YearBuilt to property age (as of latest date in dataset)
latest_year = features_df['YearBuilt'].max()
features_df['PropertyAge'] = latest_year - features_df['YearBuilt']

# --- House Size Indicator ---
# Combine building and land dimensions into one general property size measure
features_df["TotalArea"] = features_df["Landsize"] + features_df["BuildingArea"]
features_df["Log_TotalArea"] = np.log1p(features_df["TotalArea"])

**Feature: Is_CBD**

**Description:**
The Is_CBD column is a binary feature that indicates whether a property is located within 5 kilometers of Melbourne’s Central Business District (CBD).

- 1 → Property is within 5 km of the CBD (central area)
- 0 → Property is beyond 5 km from the CBD

**Rationale:**
Property prices in Melbourne are strongly influenced by proximity to the city center. Houses located closer to the CBD typically have:

- Better access to jobs, transport, and amenities
- Higher demand and limited land availability
Consequently, higher prices per square meter. By converting the continuous Distance variable into this binary flag, we capture the “CBD premium”, the price boost associated with being in or near the central urban zone.

**Why it’s important:**
- Adds a non-linear perspective to the model that pure distance may not fully capture.
- Helps the model distinguish between prime urban locations and suburban/outer areas, even if two properties are similar in size or age.
- Works well alongside the continuous Distance feature to represent both the overall distance effect and the central location effect.

In [ ]:
# Create is_CBD binary feature
features_df['Is_CBD'] = (features_df['Distance'] < 5).astype(int)

In [ ]:
features_df.head()

In [ ]:
has_nan = features_df.isna().any().any()
print("Contains NaN values:", has_nan)

In [ ]:
cleaned_df = features_df.copy()
cleaned_df = cleaned_df.drop(columns=['BuildingArea','Landsize','YearBuilt','TotalArea']) # Drop transformed columns
cleaned_df.to_csv('data/melbourne_housing_cleaned_featEngr.csv', index=False)

In [ ]:
cleaned_df.head()

## Feature Selection
In this section, we aim to identify which feature selection technique performs best when predicting house prices using various regression models.
Feature selection helps reduce noise, remove redundant variables, and improve both model interpretability and generalization performance. 
### Methodology

**Target and Features**
The target variable is Price (log-transformed using np.log1p to reduce skewness).
All other variables are used as predictors (X).

**Data Splitting and Evaluation**
We applied k-fold cross-validation (k=5) to obtain robust and unbiased performance estimates.
The evaluation metric used is the R² score, which measures how well the model explains the variance in the target variable.

**Feature Selection Techniques Tested**
- Variance Threshold: Removes features with low variance, assuming they carry little information.
- ANOVA F-test: Selects features most linearly correlated with the target variable.
- Mutual Information: Captures both linear and non-linear dependencies between features and the target.
- RFECV (Recursive Feature Elimination with Cross-Validation): Iteratively removes the least important features based on model coefficients and re-evaluates performance.
- Sequential Forward Selection: Gradually adds features that improve model performance, using cross-validation to validate each step.

**Regression Models Compared**
- Linear Regression
- Lasso Regression
- Decision Tree Regressor
- Random Forest Regressor
- Gradient Boosting Regressor

Each feature selection method was wrapped in a Pipeline with scaling and the chosen regression model, then evaluated via cross-validation.

In [ ]:
exp_df = encoded_df.copy()
# --- Property Age ---
# Convert YearBuilt to property age (as of latest date in dataset)
latest_year = exp_df['YearBuilt'].max()
exp_df['PropertyAge'] = latest_year - exp_df['YearBuilt']

# --- House Size Indicator ---
# Combine building and land dimensions into one general property size measure
exp_df["TotalArea"] = exp_df["Landsize"] + exp_df["BuildingArea"]
exp_df["Log_TotalArea"] = np.log1p(exp_df["TotalArea"])

# Create is_CBD binary feature
exp_df['Is_CBD'] = (exp_df['Distance'] < 5).astype(int)

exp_df.drop(columns=['BuildingArea','Landsize','YearBuilt','TotalArea'])

In [ ]:
# --- Dataset ---
selection_df = exp_df.copy()
y = np.log1p(selection_df['Price'])
X = selection_df.drop(columns=['Price'])

# --- Define cross-validation ---
cv = KFold(n_splits=5, shuffle=True, random_state=0)

# --- Define feature selection methods ---
def get_feature_selectors(base_model):
    return {
        "VarianceThreshold": Pipeline([
            ("scale", StandardScaler()),
            ("select", VarianceThreshold(threshold=0.01)),
            ("reg", base_model)
        ]),
        "ANOVA F-test": Pipeline([
            ("scale", StandardScaler()),
            ("select", SelectKBest(score_func=f_regression, k=12)),
            ("reg", base_model)
        ]),
        "Mutual Info": Pipeline([
            ("scale", StandardScaler()),
            ("select", SelectKBest(score_func=mutual_info_regression, k=12)),
            ("reg", base_model)
        ]),
        "RFECV": Pipeline([
            ("scale", StandardScaler()),
            ("rfecv", RFECV(LinearRegression(), step=1, cv=5, scoring="r2")),
            ("reg", base_model)
        ]),
        "Forward Selection": Pipeline([
            ("scale", StandardScaler()),
            ("forward_selection", SequentialFeatureSelector(
                LinearRegression(),
                n_features_to_select=10,
                direction='forward',
                scoring='r2',
                cv=5,
                n_jobs=-1
            )),
            ("reg", base_model)
        ])
    }

# --- Define models ---
models = {
    "LinearRegression": LinearRegression(),
    "Lasso": Lasso(alpha=0.001, max_iter=5000),
    "DecisionTree": DecisionTreeRegressor(random_state=0),
    "RandomForest": RandomForestRegressor(n_estimators=200, random_state=0, n_jobs=-1),
    "GradientBoosting": GradientBoostingRegressor(random_state=0)
}

# --- Evaluate all combinations ---
results = []

for model_name, model in models.items():
    print(f"\n=== Evaluating model: {model_name} ===")
    selectors = get_feature_selectors(model)
    
    for sel_name, pipe in selectors.items():
        scores = cross_val_score(pipe, X, y, cv=cv, scoring="r2", n_jobs=-1)
        results.append({
            "Model": model_name,
            "Feature_Selection": sel_name,
            "Mean_R2": np.mean(scores),
            "Std_R2": np.std(scores)
        })

# --- Results DataFrame ---
results_df = pd.DataFrame(results).sort_values(["Mean_R2"], ascending=False).reset_index(drop=True)

print("\nCross-validated results:")
print(results_df)

**Key Observations:**
- Ensemble methods (Random Forest, Gradient Boosting) achieved the highest R² scores (~0.77–0.78).
- Simple methods like Variance Threshold performed surprisingly well, suggesting most features are informative.
- RFECV also performed strongly across models, confirming its ability to adaptively eliminate redundant predictors.
- Complex selectors (Mutual Information, Forward Selection) did not outperform simpler ones, possibly due to dataset structure or limited non-linear dependencies.

**Conclusion**
Based on cross-validated R² performance across all models:
Variance Threshold and RFECV consistently provided the best results.

> **Note**: The feature selection was done on separate cleaned dataset where categorical columns where encoded and the dataset was mostly cleaned.

## Preprocessing Pipeline

In [ ]:
from sklearn.impute import SimpleImputer
# ==============================
# 1. Custom Frequency Encoder
# ==============================
from utils import FrequencyEncoder

# ==============================
# 2. Column Setup
# ==============================
freq_cols = ["Suburb", "SellerG", "CouncilArea"]
onehot_cols = ["Type", "Method", "Regionname"]

numeric_cols = [
    "Distance", "Bedroom", "Bathroom", "Car",
    "Propertycount", "Latitude", "Longtitude",
    "PropertyAge", "Log_Landsize", "Log_BuildingArea", "Log_TotalArea", "Is_CBD"
]

# ==============================
# 3. Preprocessing Pipelines
# ==============================
# Frequency encoding pipeline
freq_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("freq_enc", FrequencyEncoder(cols=freq_cols))
])

# One-hot encoding pipeline
onehot_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False))
])

# Numeric scaling pipeline
num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Combine all preprocessors
preprocessor = ColumnTransformer([
    ("freq", freq_pipe, freq_cols),
    ("onehot", onehot_pipe, onehot_cols),
    ("num", num_pipe, numeric_cols)
])

# ==============================
# 4. Full Feature Selection + Model Pipeline
# ==============================
full_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("variance", VarianceThreshold(threshold=0.01)),
    ("rfecv", "passthrough"), # RFECV(LinearRegression(), step=1, cv=5, scoring="r2", n_jobs=-1)
    ("model", RandomForestRegressor(n_estimators=200, random_state=0, n_jobs=-1))
])

# Save
joblib.dump(full_pipeline, "utils/melbourne_pipeline.pkl")
print(f"Pipeline Saved!!")

**Inspecting our preprocess step with different treshhold values**

In [ ]:
y = np.log1p(cleaned_df["Price"])
X = cleaned_df.drop(columns=["Price"])

full_pipeline.fit(X, y)

# Extract the preprocessor from the pipeline
preprocessor = full_pipeline.named_steps["preprocess"]

# Get feature names after preprocessing
feature_names = preprocessor.get_feature_names_out()
print(f"Total features before VarianceThreshold: {len(feature_names)}")

var_thresh = full_pipeline.named_steps["variance"]

mask = var_thresh.get_support()  # True = kept, False = dropped

# Kept features
kept_features = np.array(feature_names)[mask]

# Dropped features
dropped_features = np.array(feature_names)[~mask]

print(f"Features kept: {len(kept_features)}")
print(f"Features dropped: {len(dropped_features)}")

print("\nDropped columns due to low variance:")
print(dropped_features)

#### Insights
| Threshhold | Features Kept |Features Dropped | Dropped Columns |
|----------|----------|----------------|--------|
| 0.01 | 24 | 4 | onehot__Method_SA, onehot__Regionname_Eastern Victoria, onehot__Regionname_Northern Victoria, onehot__Regionname_Western Victoria |
| 0.05 | 23 | 5 | onehot__Method_SA, onehot__Regionname_Eastern Victoria onehot__Regionname_Northern Victoria,onehot__Regionname_South-Eastern Metropolitan onehot__Regionname_Western Victoria |
| 0.1 | 21 | 7 | onehot__Type_t, onehot__Method_SA, onehot__Method_VB, onehot__Regionname_Eastern Victoria, onehot__Regionname_Northern Victoria, onehot__Regionname_South-Eastern Metropolitan, onehot__Regionname_Western Victoria |